**Prérequis : `04_modelisation_churn.ipynb`** et **`05_modelisation_clv.ipynb`**
exécutés (`data/model/model.joblib` et `data/model/model_clv.joblib` disponibles).

---

# TP8 — Implémentation et mise en exploitation : de deux scores à une liste priorisée

**Cas d'usage :** éditeur SaaS B2B -- churn + valeur vie client.

**Portée de ce notebook [C6].** Les notebooks 04 et 05 s'arrêtent chacun à un
modèle sérialisé et une métrique. Ce notebook répond à la question métier réelle :
**qu'est-ce qu'un CSM voit, concrètement, le premier jour du mois ?** Il combine
les deux scores en une table de priorisation, esquisse une API de scoring, et
exporte un jeu d'exemple au format déjà défini pour le CRM.

**Décisions appliquées, source unique de vérité :
`Livrables/releve_decision_pdc3_churn_saas.docx`** (point de contact métier n°3,
simulé -- voir avertissement dans `cr_atelier_cadrage_churn_saas_v3.docx`) :

| # | Décision | Conséquence pour ce notebook |
|---|---|---|
| D3 | Le modèle reste un outil de priorisation, aucune action automatique | La sortie est une recommandation lue par un CSM, jamais exécutée seule -- confirmé §4 |
| D6 | Finalité unique : priorisation des actions de rétention CS | Aucune autre colonne que celles utiles à cette finalité n'est exportée -- confirmé §6 |
| D7 | `commentaire_csm` jamais exposé | Absent des colonnes exportées -- vérifié §6 |
| D9 | Seuil de classification fixé pour garantir un **rappel ≥ 80%** sur `churn` (pas un seuil de coût, pas 0.5 par défaut) | Calculé en §2 sur le jeu de test, remplace le seuil de coût pur de `04_modelisation_churn.ipynb` §6 |
| D10 | Capacité opérationnelle CS retenue : **150 comptes priorisés par cycle mensuel** | Contrainte de la règle de priorité D14, §3 |
| D11 | Catalogue d'actions à 3 niveaux (appel personnalisé / email ciblé / surveillance passive) | Texte des recommandations en §3 |
| D14 | Règle de priorité : Haute = signalés (p≥seuil D9) triés par perte attendue, dans la limite de D10 ; Moyenne = signalés restants ; Basse = non signalés | Implémentée en §3 |

**Correction par rapport à une version antérieure de ce notebook.** Une première
version avait introduit un facteur τ (taux de succès de relance) et une capacité
de 50/mois -- des valeurs fixées par le porteur du projet en l'absence du relevé
de décision D8-D15, qui n'existait pas encore. Elles sont abandonnées au profit
des décisions D9/D10/D14 ci-dessus, qui font foi.

In [1]:
import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

from sklearn.metrics import recall_score, precision_score

GOLD_DIR = Path("../data/gold")
MODEL_DIR = Path("../data/model")

PROCESSED_AT = datetime.now(timezone.utc).isoformat()

gold = pd.read_parquet(GOLD_DIR / "clients_churn_gold.parquet")
with open(GOLD_DIR / "gold_manifest.json", "r", encoding="utf-8") as f:
    gold_manifest = json.load(f)

model_churn = joblib.load(MODEL_DIR / "model.joblib")
model_clv = joblib.load(MODEL_DIR / "model_clv.joblib")

feature_columns = gold_manifest["features_modele_principal"]

print("Modèles chargés : churn +", "CLV")

Modèles chargés : churn + CLV


## §0 — Simuler un nouveau cycle de scoring

Le batch mensuel décrit dans l'architecture cible (C7) s'applique à l'ensemble des
comptes actifs, pas seulement au jeu de test -- mais pour illustrer le comportement
sur des comptes que les modèles n'ont pas vus à l'entraînement, on utilise ici le
jeu de test des notebooks 04-05, comme s'il s'agissait du cycle du mois. La colonne
`churn` (issue réelle) est conservée pour calibrer le seuil D9 sur ce cycle -- elle
n'est bien sûr jamais disponible au moment réel du scoring d'un compte actif.

In [2]:
nouveau_cycle = gold[gold["split"] == "test"].copy()
X_nouveau = nouveau_cycle[feature_columns]

print(f"Comptes à scorer ce cycle : {len(nouveau_cycle)}")

Comptes à scorer ce cycle : 1000


## §1 — Fonction de scoring batch

Perte attendue = `p(churn) × CLV_estimée` -- la formule retenue au point de contact
métier n°3 (D14), sans facteur de succès de relance : la priorisation ordonne les
comptes par enjeu économique brut, le catalogue d'actions (D11) et le protocole de
mesure d'impact (D12-D13, hors périmètre de ce notebook) sont ce qui restitue
l'efficacité réelle des relances, pas un facteur multiplicatif dans le score.

In [3]:
def scorer_batch(X: pd.DataFrame, client_ids: pd.Series) -> pd.DataFrame:
    score_churn = model_churn.predict_proba(X)[:, 1]
    valeur_vie_estimee = np.expm1(model_clv.predict(X))
    valeur_vie_estimee = np.clip(valeur_vie_estimee, 0, None)

    perte_attendue = score_churn * valeur_vie_estimee  # D14 : p(churn) x CLV

    sortie = pd.DataFrame({
        "client_id": client_ids.values,
        "score_churn": np.round(score_churn, 3),
        "valeur_vie_estimee_eur": np.round(valeur_vie_estimee, 0),
        "perte_attendue_eur": np.round(perte_attendue, 0),
    })
    return sortie

resultats = scorer_batch(X_nouveau, nouveau_cycle["client_id"])
resultats.sort_values("perte_attendue_eur", ascending=False).head(10)

,client_id,score_churn,valeur_vie_estimee_eur,perte_attendue_eur
766,CLI-003063,0.868,506554.0,439785.0
851,CLI-004433,0.902,413519.0,372937.0
116,CLI-003537,0.489,705309.0,344707.0
538,CLI-000778,0.608,512155.0,311227.0
527,CLI-000791,0.936,306276.0,286757.0
71,CLI-003591,0.261,1063278.0,277926.0
652,CLI-000678,0.487,522872.0,254608.0
561,CLI-002305,0.360,703101.0,253110.0
707,CLI-001526,0.674,368005.0,247866.0
358,CLI-002470,0.615,377605.0,232151.0


## §2 — Seuil de décision (D9) et capacité CS (D10)

**D9** -- le seuil de classification est calculé sur ce cycle pour garantir un
**rappel ≥ 80%** sur `churn` : on prend le seuil le plus élevé (donc la meilleure
précision possible) qui satisfait encore cette contrainte de rappel, plutôt qu'un
seuil de coût pur (approche abandonnée, cf. `04_modelisation_churn.ipynb` §6) ou
le défaut de 0.5.

**D10** -- capacité opérationnelle CS retenue : 150 comptes priorisés par cycle
mensuel (hypothèse de travail actée au point de contact métier n°3, ≈ 5 CSM à 30
comptes/mois).

In [4]:
RAPPEL_CIBLE_D9 = 0.80
CAPACITE_CSM_D10 = 150

y_true = nouveau_cycle["churn"].to_numpy()
proba = resultats["score_churn"].to_numpy()

seuils_candidats = np.unique(proba)
seuils_valides = [s for s in seuils_candidats if recall_score(y_true, proba >= s) >= RAPPEL_CIBLE_D9]

SEUIL_D9 = max(seuils_valides) if seuils_valides else 0.0
rappel_au_seuil = recall_score(y_true, proba >= SEUIL_D9)
precision_au_seuil = precision_score(y_true, proba >= SEUIL_D9)

print(f"Seuil D9 (rappel >= {RAPPEL_CIBLE_D9:.0%}) : {SEUIL_D9:.3f}")
print(f"Rappel obtenu    : {rappel_au_seuil:.3f}")
print(f"Précision obtenue : {precision_au_seuil:.3f}")
print(f"Comptes signalés (p >= seuil D9) : {(proba >= SEUIL_D9).sum()} sur {len(proba)}")

Seuil D9 (rappel >= 80%) : 0.282
Rappel obtenu    : 0.800
Précision obtenue : 0.627
Comptes signalés (p >= seuil D9) : 357 sur 1000


## §3 — Règle de priorité (D14) et catalogue d'actions (D11)

- **Haute** -- comptes signalés (p ≥ seuil D9), triés par perte attendue
  décroissante, dans la limite de la capacité D10 (150). Action D11 : appel
  personnalisé du CSM référent sous 5 jours ouvrés.
- **Moyenne** -- comptes signalés restants (au-delà de la capacité). Action D11 :
  email ciblé et proposition d'un point d'usage.
- **Basse** -- comptes non signalés (p < seuil D9). Action D11 : surveillance
  passive, sans action dédiée.

In [5]:
resultats["signale_D9"] = resultats["score_churn"] >= SEUIL_D9

signales = resultats[resultats["signale_D9"]].sort_values("perte_attendue_eur", ascending=False)
client_ids_haute = set(signales.head(CAPACITE_CSM_D10)["client_id"])

def assigner_priorite(row):
    if not row["signale_D9"]:
        return "Basse"
    return "Haute" if row["client_id"] in client_ids_haute else "Moyenne"

def recommander_action(priorite):
    return {
        "Haute": "Appel personnalisé du CSM référent sous 5 jours ouvrés",
        "Moyenne": "Email ciblé et proposition d'un point d'usage",
        "Basse": "Surveillance passive, sans action dédiée",
    }[priorite]

resultats["priorite"] = resultats.apply(assigner_priorite, axis=1)
resultats["action_recommandee"] = resultats["priorite"].apply(recommander_action)

print(resultats["priorite"].value_counts())
print()
print(f"Comptes en priorité Haute : {(resultats['priorite'] == 'Haute').sum()} "
      f"(plafonné à la capacité D10 = {CAPACITE_CSM_D10}, même si "
      f"{resultats['signale_D9'].sum()} comptes sont signalés par le seuil D9)")

priorite
Basse      643
Moyenne    207
Haute      150
Name: count, dtype: int64

Comptes en priorité Haute : 150 (plafonné à la capacité D10 = 150, même si 357 comptes sont signalés par le seuil D9)


### Cas concret : risque élevé/faible valeur contre risque modéré/forte valeur

Lequel chaque logique retient-elle -- le seuil seul (D9) ou la priorité complète
(D14, seuil + capacité + valeur) ?

In [6]:
candidat_risque_eleve_faible_valeur = resultats.loc[
    (resultats["score_churn"] > 0.7)
    & (resultats["valeur_vie_estimee_eur"] < resultats["valeur_vie_estimee_eur"].quantile(0.25))
].sort_values("score_churn", ascending=False).head(1)

candidat_risque_modere_forte_valeur = resultats.loc[
    (resultats["score_churn"].between(0.15, 0.35))
    & (resultats["valeur_vie_estimee_eur"] > resultats["valeur_vie_estimee_eur"].quantile(0.90))
].sort_values("valeur_vie_estimee_eur", ascending=False).head(1)

exemple = pd.concat([candidat_risque_eleve_faible_valeur, candidat_risque_modere_forte_valeur])
exemple[["client_id", "score_churn", "valeur_vie_estimee_eur", "perte_attendue_eur",
         "signale_D9", "priorite", "action_recommandee"]]

,client_id,score_churn,valeur_vie_estimee_eur,perte_attendue_eur,signale_D9,priorite,action_recommandee
996,CLI-002438,1.000,2554.0,2553.0,True,Moyenne,Email ciblé et proposition d'un point d'usage
71,CLI-003591,0.261,1063278.0,277926.0,False,Basse,"Surveillance passive, sans action dédiée"


In [7]:
for _, row in exemple.iterrows():
    print(f"{row['client_id']} -- score_churn={row['score_churn']:.2f}, "
          f"CLV estimée={row['valeur_vie_estimee_eur']:,.0f}€, "
          f"perte attendue={row['perte_attendue_eur']:,.0f}€ : "
          f"signalé D9={row['signale_D9']} -> priorité {row['priorite']}")

print()
print("Le seuil D9 seul ne dit rien de l'ordre de traitement entre deux comptes signalés --")
print("c'est la règle de priorité D14 (perte attendue + capacité) qui tranche, cohérent avec")
print("le raisonnement métier D4 (le coût d'un compte perdu compte autant que sa probabilité")
print("de partir).")

CLI-002438 -- score_churn=1.00, CLV estimée=2,554€, perte attendue=2,553€ : signalé D9=True -> priorité Moyenne
CLI-003591 -- score_churn=0.26, CLV estimée=1,063,278€, perte attendue=277,926€ : signalé D9=False -> priorité Basse

Le seuil D9 seul ne dit rien de l'ordre de traitement entre deux comptes signalés --
c'est la règle de priorité D14 (perte attendue + capacité) qui tranche, cohérent avec
le raisonnement métier D4 (le coût d'un compte perdu compte autant que sa probabilité
de partir).


## §4 — Rappel explicite : jamais une action automatique (D3, art. 22 RGPD)

`action_recommandee` est un texte de recommandation à destination d'un humain --
on vérifie qu'aucune formulation ne suggère une exécution automatique, avant tout
export.

In [8]:
# Aucune des 3 actions du catalogue D11 ne doit décrire une action irréversible ou
# significative (résiliation, downgrade) exécutée sans intervention humaine -- seul
# ce risque est couvert par l'art. 22, pas le simple envoi d'un email de relance.
MOTS_INTERDITS = [
    "automatiquement exécuté", "résiliation appliquée", "downgrade appliqué",
    "sans validation", "sans intervention humaine",
]
violations = [a for a in resultats["action_recommandee"].unique()
              if any(mot in a.lower() for mot in MOTS_INTERDITS)]
assert not violations, f"Formulation à risque détectée : {violations}"

print("Catalogue d'actions D11 vérifié :")
for a in resultats["action_recommandee"].unique():
    print(" -", a)
print()
print("Vérification D3 / art. 22 : aucune action ne résilie ni ne modifie le contrat")
print("automatiquement -- 'Haute' et 'Basse' impliquent respectivement un CSM humain et")
print("une absence d'action ; 'Moyenne' est un email de relance, pas une décision sur le")
print("compte -- OK")

Catalogue d'actions D11 vérifié :
 - Email ciblé et proposition d'un point d'usage
 - Surveillance passive, sans action dédiée
 - Appel personnalisé du CSM référent sous 5 jours ouvrés

Vérification D3 / art. 22 : aucune action ne résilie ni ne modifie le contrat
automatiquement -- 'Haute' et 'Basse' impliquent respectivement un CSM humain et
une absence d'action ; 'Moyenne' est un email de relance, pas une décision sur le
compte -- OK


## §5 — Esquisse d'API de scoring

Le pipeline réel de ce projet est un **batch mensuel**, pas un service temps réel
(architecture cible, C7) -- il n'y a donc pas de serveur à faire tourner dans ce
notebook. L'esquisse ci-dessous illustre malgré tout le contrat qu'une API de
scoring exposerait si un jour un cas d'usage temps réel l'exigeait (ex. score à la
demande depuis le CRM) -- **non exécutée ici**, à seule fin de documentation :

```python
# Esquisse -- non exécutée dans ce notebook (pas de serveur requis, cf. C7)
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

class CompteAScorer(BaseModel):
    client_id: str
    # ... les 30 features de feature_columns_modele_principal, mêmes noms/types
    # que dans data/gold/gold_manifest.json -- jamais commentaire_csm.

class ScoreRetourne(BaseModel):
    client_id: str
    score_churn: float
    valeur_vie_estimee_eur: float
    priorite: str
    action_recommandee: str

@app.post("/scoring/compte", response_model=ScoreRetourne)
def scorer_un_compte(compte: CompteAScorer) -> ScoreRetourne:
    # Réutilise scorer_batch() sur un DataFrame à une ligne -- même fonction,
    # même modèles sérialisés, aucune logique dupliquée entre batch et API.
    ...
```

La fonction `scorer_batch` définie en §1 est déjà le seul point d'entrée métier --
une API n'ajouterait qu'une couche de transport HTTP par-dessus, pas une nouvelle
logique.

## §6 — Export au format CRM (contrat déjà défini en architecture cible)

Colonnes strictement limitées à celles listées dans l'architecture (slide 5) --
aucune feature brute, aucun champ texte libre.

In [9]:
COLONNES_EXPORT_CRM = ["client_id", "score_churn", "valeur_vie_estimee_eur", "priorite", "action_recommandee"]
export_crm = resultats[COLONNES_EXPORT_CRM].copy()

colonnes_interdites = set(gold.columns) - {"client_id"}
assert not (colonnes_interdites & set(export_crm.columns)), "Une colonne brute de Gold s'est glissée dans l'export"
assert "commentaire_csm" not in export_crm.columns

export_path = MODEL_DIR / "scoring_exemple_cycle.parquet"
export_crm.to_parquet(export_path, index=False)

print(f"Export CRM -- {len(export_crm)} lignes, {len(export_crm.columns)} colonnes : {list(export_crm.columns)}")
print("Fichier :", export_path)
export_crm.sort_values("valeur_vie_estimee_eur", ascending=False).head(5)

Export CRM -- 1000 lignes, 5 colonnes : ['client_id', 'score_churn', 'valeur_vie_estimee_eur', 'priorite', 'action_recommandee']
Fichier : ..\data\model\scoring_exemple_cycle.parquet


,client_id,score_churn,valeur_vie_estimee_eur,priorite,action_recommandee
808,CLI-001433,0.003,1450723.0,Basse,"Surveillance passive, sans action dédiée"
348,CLI-003214,0.110,1244274.0,Basse,"Surveillance passive, sans action dédiée"
37,CLI-002793,0.002,1229087.0,Basse,"Surveillance passive, sans action dédiée"
71,CLI-003591,0.261,1063278.0,Basse,"Surveillance passive, sans action dédiée"
427,CLI-002050,0.178,1008344.0,Basse,"Surveillance passive, sans action dédiée"


## §7 — Manifeste et vérification finale

In [10]:
scoring_manifest = {
    "couche": "model",
    "sous_etape": "implementation_scoring",
    "processed_at_utc": PROCESSED_AT,
    "modeles_utilises": {
        "churn": str(MODEL_DIR / "model.joblib"),
        "clv": str(MODEL_DIR / "model_clv.joblib"),
    },
    "regle_decision": {
        "source": "Livrables/releve_decision_pdc3_churn_saas.docx (D9, D10, D11, D14)",
        "seuil_D9_rappel_cible": RAPPEL_CIBLE_D9,
        "seuil_D9_valeur": round(float(SEUIL_D9), 3),
        "rappel_obtenu": round(float(rappel_au_seuil), 3),
        "precision_obtenue": round(float(precision_au_seuil), 3),
        "capacite_csm_D10": CAPACITE_CSM_D10,
        "formule_priorite_D14": "perte_attendue = score_churn x valeur_vie_estimee_eur, sans facteur tau",
        "catalogue_actions_D11": {
            "Haute": "Appel personnalisé du CSM référent sous 5 jours ouvrés",
            "Moyenne": "Email ciblé et proposition d'un point d'usage",
            "Basse": "Surveillance passive, sans action dédiée",
        },
    },
    "colonnes_export_crm": COLONNES_EXPORT_CRM,
    "decisions_appliquees": ["D3", "D6", "D7", "D9", "D10", "D11", "D14"],
    "decisions_hors_perimetre": {
        "D8": "critère d'acceptation du modèle -- concerne 04_modelisation_churn.ipynb, pas ce notebook",
        "D12_D13": "groupe témoin et cadence de reporting -- mesure d'impact post-déploiement (C8/C9), pas encore de notebook dédié",
        "D15": "critère de recette (corrélation de rang) -- concerne un futur notebook de recette, pas encore créé",
    },
    "contrat_source": "Livrables/architecture_donnees_churn_saas.pptx, slide 5",
    "fichier_exemple": str(export_path),
}
with open(MODEL_DIR / "scoring_manifest.json", "w", encoding="utf-8") as f:
    json.dump(scoring_manifest, f, ensure_ascii=False, indent=2)

verif = pd.DataFrame([
    {"vérification": "Comptes scorés ce cycle", "résultat": len(resultats)},
    {"vérification": "Seuil D9 (rappel >= 80%)", "résultat": round(float(SEUIL_D9), 3)},
    {"vérification": "Rappel obtenu au seuil D9", "résultat": round(float(rappel_au_seuil), 3)},
    {"vérification": "Comptes en priorité Haute (<= capacité D10)", "résultat": int((resultats["priorite"] == "Haute").sum())},
    {"vérification": "Colonnes exportées vers le CRM", "résultat": list(export_crm.columns)},
    {"vérification": "commentaire_csm absent de l'export", "résultat": "commentaire_csm" not in export_crm.columns},
    {"vérification": "Aucune formulation d'action automatique détectée", "résultat": not violations},
    {"vérification": "Manifeste de scoring écrit", "résultat": (MODEL_DIR / "scoring_manifest.json").exists()},
])
verif

,vérification,résultat
0,Comptes scorés ce cycle,1000
1,Seuil D9 (rappel >= 80%),0.282
2,Rappel obtenu au seuil D9,0.8
3,Comptes en priorité Haute (<= capacité D10),150
4,Colonnes exportées vers le CRM,"[client_id, score_churn, valeur_vie_estimee_eu..."
5,commentaire_csm absent de l'export,True
6,Aucune formulation d'action automatique détectée,True
7,Manifeste de scoring écrit,True


## Journal de bord — Synthèse TP8 (Implémentation & scoring)

- **Deux modèles combinés, pas juxtaposés** : le score de churn et la valeur vie
  client estimée forment une seule métrique de priorisation (`perte_attendue_eur`
  = `p(churn) × CLV`, D14) -- pas deux scores livrés séparément à charge pour le
  CSM de les recouper lui-même.
- **Seuil D9 calculé, pas choisi à vue** : le seuil qui maximise la précision sous
  contrainte d'un rappel ≥ 80% est recherché sur le cycle de test, remplaçant le
  seuil de coût pur exploré dans `04_modelisation_churn.ipynb` §6 -- une
  décision actée au point de contact métier n°3 (D9), pas improvisée ici.
- **Capacité D10 intégrée dans la règle de priorité (D14)**, pas seulement dans le
  score : un compte signalé par le seuil D9 n'est "priorité Haute" que s'il
  entre dans les 150 comptes de plus forte perte attendue -- ce qui rend la liste
  directement actionnable par une équipe CS à capacité limitée.
- **Catalogue d'actions D11 relié à la priorité**, pas une étiquette abstraite :
  chaque niveau correspond à une consigne de travail concrète pour un CSM.
- **Limite observée, à assumer plutôt qu'à corriger unilatéralement** : parce que
  D14 ne classe par perte attendue **qu'à l'intérieur** des comptes déjà signalés
  par D9, un compte à risque modéré mais très forte CLV (ex. `CLI-003591`,
  p=0.26, CLV=1,06 M€) peut rester en priorité Basse s'il n'atteint pas le seuil
  D9 -- même si sa perte attendue dépasserait largement celle de comptes classés
  Haute. C'est une conséquence directe de D9 (porte d'entrée par le rappel) suivi
  de D14 (tri par valeur), pas un bug -- mais un point à signaler si la
  soutenance interroge ce choix plutôt qu'un tri global par perte attendue seule.
- **Correction assumée** : une version antérieure de ce notebook avait introduit
  un facteur τ et une capacité de 50/mois fixés par le porteur du projet, avant
  l'existence du relevé de décision du point de contact métier n°3. Abandonnés au
  profit de D9/D10/D14, qui font foi.
- **Aucune décision automatisée** (D3, art. 22 RGPD) : vérifié explicitement par
  une assertion sur le texte des recommandations, pas seulement affirmé en
  commentaire.
- **Hors périmètre de ce notebook, explicitement listé dans le manifeste** : D8
  (critère d'acceptation du modèle, concerne le notebook 04), D12-D13 (groupe
  témoin et reporting trimestriel, C8/C9), D15 (critère de recette, futur
  notebook dédié).
- **Ce notebook s'arrête ici.** L'architecture cible détaillée est déjà couverte
  par `Livrables/architecture_donnees_churn_saas.pptx` [C7] ; la mesure d'impact
  post-déploiement et le ré-entraînement restent à couvrir [C8-C9].
